In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "adlproyecto12";
create widget text catalog_name default "catalog_movies";

In [0]:
storageName = dbutils.widgets.get("storageName")

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore-proy@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
%sql
DROP CATALOG IF EXISTS ${catalog_name} CASCADE;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ${catalog_name}
MANAGED LOCATION 'abfss://metastore-proy@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
%sql
DROP SCHEMA IF EXISTS ${catalog_name}.raw;
DROP SCHEMA IF EXISTS ${catalog_name}.bronze;
DROP SCHEMA IF EXISTS ${catalog_name}.silver;
DROP SCHEMA IF EXISTS ${catalog_name}.golden;

In [0]:
#Borra el archivos y subcarpetas de los contenedores.
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ${catalog_name}.raw;
CREATE SCHEMA IF NOT EXISTS ${catalog_name}.bronze;
CREATE SCHEMA IF NOT EXISTS ${catalog_name}.silver;
CREATE SCHEMA IF NOT EXISTS ${catalog_name}.golden;

###Tablas Bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog_name}.bronze.movies (
movie_id integer,
title string,
genres string,
language string,
user_score float,
runtime_hour integer,
runtime_min integer,
release_date date,
vote_count integer,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/movies"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog_name}.bronze.moviemoreinfo ( 
movie_id integer,
runtime string,
budget string,
revenue string,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/moviemoreinfo"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog_name}.bronze.moviedetails (
  movie_id integer,
  director string,
  top_billed string,
  budget_usd string,
  revenue_usd string,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/moviedetails"

###Tablas Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog_name}.silver.movies_transformed ( 
movie_id integer,
title string,
genres string,
language string,
user_score float,
release_date date,
vote_count integer,
vote_category string,
release_year integer,
total_minutos integer,
director string,
top_billed string,
budget decimal(36,2),
revenue decimal(36,2),
profit decimal(36,2),
status_ticket string,
slv_date timestamp
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/movies_transformed"

###Tablas Golden

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog_name}.golden.golden_movies (
  movie_id integer,
  title string,
  genres string,
  language string,
  user_score float,
  release_date date,
  vote_count integer,
  vote_category string,
  release_year integer,
  total_minutos integer,
  director string,
  top_billed string,
  budget decimal(36,2),
  revenue decimal(36,2),
  profit decimal(36,2),
  status_ticket string
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/golden_movies"